# ChaoSSM — Phase 19: closed-form ridge readout vs gradient descent

Addresses **Reviewer 2, comment 7**: *does using a closed-form ridge regression for the readout
(standard in ESNs) instead of gradient descent change any regime comparisons or the
effective-rank/accuracy relationship?*

We take the **identical frozen reservoirs** used throughout the paper — ordered (λ≈−0.30), edge
(λ≈0⁻), deep (λ≈+0.30), linear control, and a leaky-ESN baseline (Reviewer 2 comment 6) — on the
canonical logistic forecast task, and fit the readout two ways:

1. **GD**: a linear readout over `LayerNorm(h)` trained by gradient descent (the paper's method).
2. **Ridge**: closed-form ridge classification on collected states, `W = (HᵀH + αI)⁻¹ HᵀY` with
   one-hot targets, prediction `argmax(HW)` — the textbook ESN readout.

Both see the same frozen features up to the LayerNorm affine (identity for ridge, trainable for
GD). For each arm we also report the effective rank (participation ratio) and memory capacity, so
the effective-rank/accuracy relationship can be compared across the two readout methods.

Runtime: ~15–20 min on a T4 (the GD arm dominates; ridge is instant). Resumable.

## Setup

In [ ]:
import os, json, math, time, warnings
from dataclasses import dataclass, field
import numpy as np, torch, torch.nn as nn, torch.nn.functional as F
import matplotlib; matplotlib.use("Agg"); import matplotlib.pyplot as plt
import pandas as pd
warnings.filterwarnings("ignore")
DEVICE=torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("torch",torch.__version__,"| device:",DEVICE)
if DEVICE.type=="cuda": print("gpu:",torch.cuda.get_device_name(0))

def setup_results_dir(project="ChaoSSM",phase="phase19_ridge_readout"):
    base=None
    try:
        from google.colab import drive
        try: drive.mount("/content/drive",force_remount=False)
        except Exception as e1:
            print(f"[warn] drive.mount failed ({type(e1).__name__}); retrying force_remount=True")
            drive.mount("/content/drive",force_remount=True)
        base=f"/content/drive/My Drive/{project}/results/{phase}"
    except Exception as e:
        base=os.path.abspath(f"/content/{project}_results/{phase}") if os.path.isdir("/content") \
             else os.path.abspath(f"./{project}_results/{phase}")
        print("="*78)
        print(f"[WARNING] Google Drive NOT mounted ({type(e).__name__}). Results -> {base}")
        print("[WARNING] That is local disk, lost on runtime recycle. Run the rescue cell or download.")
        print("="*78)
    os.makedirs(base,exist_ok=True); print("results ->",base); return base
RESULTS=setup_results_dir()
def savefig(f,n): p=os.path.join(RESULTS,n); f.savefig(p,dpi=600,bbox_inches="tight"); print("saved",p)
def savetable(d,n): p=os.path.join(RESULTS,n); d.to_csv(p,index=False); print("saved",p)

torch 2.11.0+cu130 | device: cuda
gpu: Tesla T4
Mounted at /content/drive
results -> /content/drive/My Drive/ChaoSSM/results/phase19_ridge_readout


## Configuration

In [ ]:
SMOKE_TEST=False

@dataclass
class Config:
    state_dim:int=64; d_model:int=64
    steps:int=2500; batch:int=64; warmup_frac:float=0.05; lr_min_frac:float=0.1
    grad_clip:float=1.0; gamma_init:float=1.0; lr:float=1e-3
    seeds:int=5
    eval_n:int=1500; eval_seed:int=12345; fc_length:int=64; fc_bins:int=32
    r_cap_map:float=4.0
    lambda_target:dict=field(default_factory=lambda: dict(ordered=-0.30, edge=-0.02, deep=0.30))
    lambda_tol:float=0.05; calib_steps:int=200; calib_warm:int=60
    arms:tuple=("ordered","edge","deep","linear-control","leaky-esn")
    leak_rate:float=0.3; esn_rho:float=0.95
    ridge_alpha:float=1e-2; ridge_T:int=400; ridge_streams:int=400
CFG=Config()
if SMOKE_TEST:
    CFG.steps=40; CFG.seeds=2; CFG.eval_n=60; CFG.calib_steps=30; CFG.calib_warm=10
    CFG.arms=("ordered","edge","linear-control"); CFG.ridge_streams=40
    print("*** SMOKE TEST — results meaningless ***")
print("arms:",CFG.arms,"| seeds:",CFG.seeds)

arms: ('ordered', 'edge', 'deep', 'linear-control', 'leaky-esn') | seeds: 5


## Cells, calibration — verbatim from Phase 14, plus the leaky-ESN arm

In [ ]:
def triangle_fold(x): return 1.0-torch.abs(torch.remainder(x,2.0)-1.0)

class ChaosCell(nn.Module):
    def __init__(self,cfg,regime):
        super().__init__(); self.N=cfg.state_dim; self.regime=regime; self.r_cap=float(cfg.r_cap_map)
        self.w_r=nn.Parameter(torch.zeros(self.N))
        W=torch.empty(self.N,self.N); nn.init.orthogonal_(W); self.W_mix=nn.Parameter(W*0.1)
        self.w_gamma=nn.Parameter(torch.tensor(float(math.log(math.expm1(cfg.gamma_init)))))
        self.B=nn.Parameter(torch.randn(self.N,cfg.d_model)/math.sqrt(cfg.d_model))
    def get_r(self): return self.r_cap*torch.sigmoid(self.w_r)
    def get_gamma(self): return F.softplus(self.w_gamma)
    def set_gamma(self,val):
        with torch.no_grad(): self.w_gamma.copy_(torch.tensor(float(math.log(math.expm1(max(val,1e-3))))))
    def phi(self,h):
        r=self.get_r(); u=h@self.W_mix.T+r*h*(1-h); return triangle_fold(self.get_gamma()*u)
    def step(self,h,x): return triangle_fold(self.phi(h)+x@self.B.T)
    def init_state(self,b,dev,dt): return torch.full((b,self.N),0.5,device=dev,dtype=dt)
    def freeze_recurrence(self):
        for p_ in [self.w_r,self.W_mix,self.w_gamma]: p_.requires_grad_(False)
    def freeze_input(self): self.B.requires_grad_(False)

class LinearCell(nn.Module):
    def __init__(self,cfg,**kw):
        super().__init__(); self.N=cfg.state_dim
        W=torch.empty(self.N,self.N); nn.init.orthogonal_(W); self.W_lin=nn.Parameter(W*0.9)
        self.B=nn.Parameter(torch.randn(self.N,cfg.d_model)/math.sqrt(cfg.d_model))
    def phi(self,h): return torch.tanh(h@self.W_lin.T)          # for uniform state-collection API
    def step(self,h,x): return torch.tanh(h@self.W_lin.T+x@self.B.T)
    def init_state(self,b,dev,dt): return torch.zeros(b,self.N,device=dev,dtype=dt)
    def freeze_recurrence(self): self.W_lin.requires_grad_(False)
    def freeze_input(self): self.B.requires_grad_(False)

class LeakyESNCell(nn.Module):
    def __init__(self,cfg,**kw):
        super().__init__(); self.N=cfg.state_dim; self.a=float(cfg.leak_rate)
        W=torch.randn(self.N,self.N)/math.sqrt(self.N)
        ev=torch.linalg.eigvals(W).abs().max().item()
        self.W=nn.Parameter(W*(cfg.esn_rho/max(ev,1e-9)))
        self.B=nn.Parameter(torch.randn(self.N,cfg.d_model)/math.sqrt(cfg.d_model))
    def step(self,h,x): return (1-self.a)*h+self.a*torch.tanh(h@self.W.T+x@self.B.T)
    def init_state(self,b,dev,dt): return torch.zeros(b,self.N,device=dev,dtype=dt)
    def freeze_recurrence(self): self.W.requires_grad_(False)
    def freeze_input(self): self.B.requires_grad_(False)

from torch.func import jacrev
@torch.no_grad()
def map_lyapunov(cell,cfg,seed=0,steps=None,warm=None):
    steps=steps or cfg.calib_steps; warm=warm or cfg.calib_warm; N=cfg.state_dim; dev=cell.W_mix.device
    g=torch.Generator(device=dev); g.manual_seed(seed)
    h=torch.full((N,),0.5,device=dev); Q=torch.linalg.qr(torch.randn(N,N,generator=g,device=dev))[0]
    logsum=torch.zeros(N,device=dev)
    for t in range(steps+warm):
        def stp(z):
            r=cell.get_r(); u=z@cell.W_mix.T+r*z*(1-z); return triangle_fold(cell.get_gamma()*u)
        J=jacrev(stp)(h); h=stp(h)+0.02*torch.randn(N,generator=g,device=dev); h=torch.clamp(h,0,1)
        Q,R=torch.linalg.qr(J@Q); d=torch.sign(torch.diagonal(R)); Q=Q*d; R=R*d.unsqueeze(0)
        if t>=warm: logsum=logsum+torch.log(torch.abs(torch.diagonal(R))+1e-12)
    return float(torch.sort(logsum/steps,descending=True)[0][0])

def calibrate_gamma(cell,cfg,regime,seed=0):
    target=cfg.lambda_target[regime]; lo,hi=0.01,8.0; best=None; best_err=1e9
    for _ in range(18):
        mid=math.sqrt(lo*hi); cell.set_gamma(mid); lam=map_lyapunov(cell,cfg,seed=seed)
        err=abs(lam-target)
        if err<best_err: best_err=err; best=(mid,lam)
        if lam<target: lo=mid
        else: hi=mid
        if err<cfg.lambda_tol: break
    cell.set_gamma(best[0]); return best[0],best[1],bool(best_err<cfg.lambda_tol)

def build_cell(cfg,arm,seed=0):
    lam=float("nan"); ok=True
    if arm=="linear-control": cell=LinearCell(cfg).to(DEVICE)
    elif arm=="leaky-esn":    cell=LeakyESNCell(cfg).to(DEVICE)
    else:
        cell=ChaosCell(cfg,arm).to(DEVICE); _,lam,ok=calibrate_gamma(cell,cfg,arm,seed=seed)
    cell.freeze_recurrence(); cell.freeze_input()
    return cell,lam,ok

## Task (logistic forecast, r=3.9) and the frozen-reservoir readout model

In [ ]:
def make_forecast(cfg,batch,device,gen):
    L=cfg.fc_length; r=3.9
    x0=0.1+0.8*torch.rand(batch,1,device=device,generator=gen); xs=[x0]
    for _ in range(L): xs.append(r*xs[-1]*(1-xs[-1]))
    X=torch.cat(xs,1); inp=X[:,:L]; nxt=X[:,1:L+1]
    tgt=torch.clamp((nxt*cfg.fc_bins).long(),0,cfg.fc_bins-1)
    return inp.float(),tgt

class ReservoirLM(nn.Module):
    """Canonical frozen reservoir: random frozen encoder, LayerNorm, trainable linear readout."""
    def __init__(self,cfg,cell,out_dim,ln_affine=True):
        super().__init__(); self.inp=nn.Linear(1,cfg.d_model); self.cell=cell
        self.ln=nn.LayerNorm(cfg.state_dim,elementwise_affine=ln_affine)
        self.readout=nn.Linear(cfg.state_dim,out_dim)
        for p_ in self.inp.parameters(): p_.requires_grad_(False)   # frozen drive
    def states(self,x):
        with torch.no_grad():
            e=self.inp(x.unsqueeze(-1)); B,T=e.shape[0],e.shape[1]
            h=self.cell.init_state(B,e.device,e.dtype); H=[]
            for t in range(T): h=self.cell.step(h,e[:,t]); H.append(h)
            return torch.stack(H,1)
    def features(self,x): return self.ln(self.states(x))
    def forward(self,x): return self.readout(self.features(x))

def make_lr_lambda(cfg,total):
    warm=max(1,int(cfg.warmup_frac*total))
    def fn(s):
        if s<warm: return s/warm
        p=(s-warm)/max(1,total-warm); return cfg.lr_min_frac+(1-cfg.lr_min_frac)*0.5*(1+math.cos(math.pi*p))
    return fn

@torch.no_grad()
def eval_acc(model,cfg,gen_seed,chunk=256):
    g=torch.Generator(device=DEVICE if DEVICE.type=="cuda" else "cpu"); g.manual_seed(gen_seed)
    inp,tgt=make_forecast(cfg,cfg.eval_n,DEVICE,g); model.eval(); c=tot=0.0
    for i in range(0,inp.shape[0],chunk):
        p=model(inp[i:i+chunk]).argmax(-1); t=tgt[i:i+chunk]
        c+=(p==t).float().sum().item(); tot+=t.numel()
    model.train(); return c/max(tot,1)

## Two readouts

**GD readout** trains `LayerNorm(h)` (affine) + linear layer by gradient descent, exactly as in
the paper. **Ridge readout** collects `LayerNorm(h)` features with identity affine over many
forecast streams, solves closed-form ridge classification onto one-hot targets, and predicts by
argmax. Both are evaluated on the same held-out set.

In [ ]:
def train_gd(cfg,cell,seed):
    model=ReservoirLM(cfg,cell,cfg.fc_bins,ln_affine=True).to(DEVICE)
    params=[p for p in model.parameters() if p.requires_grad]
    opt=torch.optim.AdamW(params,lr=cfg.lr,betas=(0.9,0.95))
    sched=torch.optim.lr_scheduler.LambdaLR(opt,make_lr_lambda(cfg,cfg.steps))
    gen=torch.Generator(device=DEVICE if DEVICE.type=="cuda" else "cpu"); gen.manual_seed(seed)
    div=False
    for step in range(cfg.steps):
        inp,tgt=make_forecast(cfg,cfg.batch,DEVICE,gen)
        lg=model(inp); L=F.cross_entropy(lg.reshape(-1,lg.shape[-1]),tgt.reshape(-1))
        opt.zero_grad(set_to_none=True); L.backward()
        gn=torch.nn.utils.clip_grad_norm_(params,cfg.grad_clip).item(); opt.step(); sched.step()
        if not(math.isfinite(L.item()) and math.isfinite(gn)): div=True; break
    acc=float("nan") if div else eval_acc(model,cfg,cfg.eval_seed)
    return acc,div,sum(p.numel() for p in params)

@torch.no_grad()
def train_ridge(cfg,cell,seed):
    """Closed-form ridge classification on LayerNorm(h) features (identity affine)."""
    model=ReservoirLM(cfg,cell,cfg.fc_bins,ln_affine=False).to(DEVICE)
    gen=torch.Generator(device=DEVICE if DEVICE.type=="cuda" else "cpu"); gen.manual_seed(seed)
    # collect features + one-hot targets over many streams
    inp,tgt=make_forecast(cfg,cfg.ridge_streams,DEVICE,gen)
    Feat=model.features(inp).reshape(-1,cfg.state_dim)              # (S*T, N)
    Y=F.one_hot(tgt.reshape(-1),cfg.fc_bins).float()               # (S*T, bins)
    X=torch.cat([Feat,torch.ones(Feat.shape[0],1,device=Feat.device)],1)   # bias column
    A=X.T@X+cfg.ridge_alpha*torch.eye(X.shape[1],device=X.device)
    Wr=torch.linalg.solve(A, X.T@Y)                                # (N+1, bins)
    # evaluate on held-out
    ge=torch.Generator(device=DEVICE if DEVICE.type=="cuda" else "cpu"); ge.manual_seed(cfg.eval_seed)
    ei,et=make_forecast(cfg,cfg.eval_n,DEVICE,ge)
    EF=model.features(ei).reshape(-1,cfg.state_dim)
    EX=torch.cat([EF,torch.ones(EF.shape[0],1,device=EF.device)],1)
    pred=(EX@Wr).argmax(-1); acc=float((pred==et.reshape(-1)).float().mean())
    return acc

## Effective rank (participation ratio) and memory capacity — verbatim from Phase 14

In [ ]:
@torch.no_grad()
def memory_capacity(cell,cfg,encode_fn,seed=0,T=1500,washout=200,max_delay=40):
    dev=next(p.device for p in cell.parameters()); g=torch.Generator(device=dev); g.manual_seed(seed)
    u=torch.rand(T,generator=g,device=dev); drive=encode_fn(u.view(1,-1)).squeeze(0)
    h=cell.init_state(1,dev,torch.float32).squeeze(0); H=[]
    for t in range(T):
        h=triangle_fold(cell.phi(h)+drive[t]@cell.B.T) if hasattr(cell,"W_mix") \
          else torch.tanh(h@cell.W_lin.T+drive[t]@cell.B.T) if hasattr(cell,"W_lin") \
          else cell.step(h,drive[t])
        H.append(h)
    H=torch.stack(H); Hd=H[washout:]; um=u[washout:]
    X=torch.cat([Hd,torch.ones(Hd.shape[0],1,device=dev)],1); mc=0.0
    for k in range(1,max_delay+1):
        if k>=Hd.shape[0]-2: break
        Xk=X[k:]; yk=um[:-k]
        A=Xk.T@Xk+1e-3*torch.eye(Xk.shape[1],device=dev); b=Xk.T@yk
        w=torch.linalg.lstsq(A,b.unsqueeze(1)).solution.squeeze(1); pred=Xk@w
        vp=pred-pred.mean(); vy=yk-yk.mean()
        r2=float(torch.clamp((vp@vy)**2/((vp@vp)*(vy@vy)+1e-12),0,1)); mc+=r2
    return mc

@torch.no_grad()
def effective_rank(cell,cfg,encode_fn,seed=0,n_streams=200,T=60):
    dev=next(p.device for p in cell.parameters()); g=torch.Generator(device=dev); g.manual_seed(seed)
    finals=[]
    for s in range(n_streams):
        u=torch.rand(T,generator=g,device=dev); drive=encode_fn(u.view(1,-1)).squeeze(0)
        h=cell.init_state(1,dev,torch.float32).squeeze(0)
        for t in range(T):
            h=triangle_fold(cell.phi(h)+drive[t]@cell.B.T) if hasattr(cell,"W_mix") \
              else torch.tanh(h@cell.W_lin.T+drive[t]@cell.B.T) if hasattr(cell,"W_lin") \
              else cell.step(h,drive[t])
        finals.append(h)
    M=torch.stack(finals); M=M-M.mean(0,keepdim=True); sv=torch.linalg.svdvals(M)
    return float((sv.sum()**2)/(sv.pow(2).sum()+1e-12))

## Run — resumable

In [ ]:
FRESH_START=False
CKPT=os.path.join(RESULTS,"phase19_ridge_readout.csv")
COLS=["arm","seed","lambda","acc_gd","acc_ridge","eff_rank","mc","diverged_gd","n_trainable_gd"]
if FRESH_START and os.path.exists(CKPT): os.replace(CKPT,CKPT+".bak")
if os.path.exists(CKPT):
    rows=pd.read_csv(CKPT).to_dict("records"); done={(r["arm"],int(r["seed"])) for r in rows}
    print(f"[resume] {len(rows)} runs done")
else: rows=[]; done=set(); print("[resume] fresh start")
def _flush(): pd.DataFrame(rows)[COLS].to_csv(CKPT,index=False)

todo=[(a,s) for a in CFG.arms for s in range(CFG.seeds) if (a,s) not in done]
print(f"{len(todo)} runs to go\n"); t0=time.time()
for arm,sd in todo:
    torch.manual_seed(sd); np.random.seed(sd)
    cell,lam,ok=build_cell(CFG,arm,seed=sd)
    enc=nn.Linear(1,CFG.d_model).to(DEVICE)              # frozen random encoder for MC / eff-rank
    for p_ in enc.parameters(): p_.requires_grad_(False)
    encode_fn=lambda u,_e=enc: _e(u.unsqueeze(-1))
    acc_gd,div,nt=train_gd(CFG,cell,sd)
    acc_ridge=train_ridge(CFG,cell,sd)
    er=effective_rank(cell,CFG,encode_fn,seed=sd)
    mc=memory_capacity(cell,CFG,encode_fn,seed=sd)
    rows.append(dict(arm=arm,seed=sd,**{"lambda":lam},acc_gd=acc_gd,acc_ridge=acc_ridge,
                     eff_rank=er,mc=mc,diverged_gd=div,n_trainable_gd=nt)); _flush()
    print(f"[P19] {arm:14s} s={sd} lam={lam:+.3f} GD={acc_gd:.4f} ridge={acc_ridge:.4f} "
          f"eff_rank={er:.2f} MC={mc:.2f}")
RES=pd.DataFrame(rows)[COLS]; _flush()
print(f"\nwall clock: {(time.time()-t0)/60:.1f} min")

[resume] fresh start
25 runs to go

[P19] ordered        s=0 lam=-0.274 GD=0.3959 ridge=0.4191 eff_rank=6.12 MC=4.98
[P19] ordered        s=1 lam=-0.278 GD=0.3334 ridge=0.4301 eff_rank=5.67 MC=5.48
[P19] ordered        s=2 lam=-0.274 GD=0.3887 ridge=0.4607 eff_rank=5.96 MC=5.15
[P19] ordered        s=3 lam=-0.276 GD=0.4151 ridge=0.3784 eff_rank=6.03 MC=5.44
[P19] ordered        s=4 lam=-0.277 GD=0.3543 ridge=0.3707 eff_rank=5.67 MC=5.43
[P19] edge           s=0 lam=+0.015 GD=0.3823 ridge=0.4038 eff_rank=8.38 MC=5.65
[P19] edge           s=1 lam=+0.009 GD=0.3388 ridge=0.3692 eff_rank=7.91 MC=5.96
[P19] edge           s=2 lam=+0.009 GD=0.3648 ridge=0.3681 eff_rank=8.26 MC=5.66
[P19] edge           s=3 lam=+0.017 GD=0.4019 ridge=0.3258 eff_rank=8.17 MC=5.74
[P19] edge           s=4 lam=+0.013 GD=0.3660 ridge=0.3413 eff_rank=7.93 MC=5.62
[P19] deep           s=0 lam=+0.272 GD=0.2524 ridge=0.2092 eff_rank=41.94 MC=3.26
[P19] deep           s=1 lam=+0.293 GD=0.2398 ridge=0.2194 eff_rank=41.2

## Analysis — does the regime ordering survive the readout change?

In [ ]:
g=RES.groupby("arm")
tab=pd.DataFrame({
  "lambda":g["lambda"].mean().round(3),
  "acc_GD":g["acc_gd"].mean().round(4), "acc_GD_sd":g["acc_gd"].std().round(4),
  "acc_ridge":g["acc_ridge"].mean().round(4), "acc_ridge_sd":g["acc_ridge"].std().round(4),
  "eff_rank":g["eff_rank"].mean().round(2), "MC":g["mc"].mean().round(2),
}).reindex(CFG.arms)
savetable(tab.reset_index(),"phase19_summary.csv")
print(tab.to_string()); print()
# rank-order agreement between the two readouts
from scipy.stats import spearmanr
m=tab.dropna()
rho,pv=spearmanr(m["acc_GD"],m["acc_ridge"])
print(f"Spearman(acc_GD, acc_ridge) across arms = {rho:+.3f} (p={pv:.3f})")
print("ordering by GD   :",list(m["acc_GD"].sort_values(ascending=False).index))
print("ordering by ridge:",list(m["acc_ridge"].sort_values(ascending=False).index))
print("\neffective-rank / accuracy relationship:")
print(f"  Spearman(eff_rank, acc_GD)    = {spearmanr(m['eff_rank'],m['acc_GD'])[0]:+.3f}")
print(f"  Spearman(eff_rank, acc_ridge) = {spearmanr(m['eff_rank'],m['acc_ridge'])[0]:+.3f}")
print("  (paper reports neither MC nor separation tracks accuracy across regimes)")

saved /content/drive/My Drive/ChaoSSM/results/phase19_ridge_readout/phase19_summary.csv
                lambda  acc_GD  acc_GD_sd  acc_ridge  acc_ridge_sd  eff_rank     MC
arm                                                                                
ordered         -0.276  0.3775     0.0330     0.4118        0.0374      5.89   5.30
edge             0.013  0.3707     0.0234     0.3616        0.0299      8.13   5.72
deep             0.293  0.2485     0.0084     0.2196        0.0077     41.56   3.38
linear-control     NaN  0.2442     0.0033     0.2511        0.0028      6.92  11.56
leaky-esn          NaN  0.1849     0.0027     0.1986        0.0302      3.07   6.27

Spearman(acc_GD, acc_ridge) across arms = +1.000 (p=0.000)
ordering by GD   : ['ordered', 'edge', 'deep']
ordering by ridge: ['ordered', 'edge', 'deep']

effective-rank / accuracy relationship:
  Spearman(eff_rank, acc_GD)    = -1.000
  Spearman(eff_rank, acc_ridge) = -1.000
  (paper reports neither MC nor separation trac

## Figure (600 dpi)

In [ ]:
fig,ax=plt.subplots(1,2,figsize=(12,4.6))
arms=[a for a in CFG.arms if a in tab.index]
x=np.arange(len(arms)); w=0.38
ax[0].bar(x-w/2,[tab.loc[a,"acc_GD"] for a in arms],w,yerr=[tab.loc[a,"acc_GD_sd"] for a in arms],
          capsize=3,label="GD readout",color="#4C72B0")
ax[0].bar(x+w/2,[tab.loc[a,"acc_ridge"] for a in arms],w,yerr=[tab.loc[a,"acc_ridge_sd"] for a in arms],
          capsize=3,label="ridge readout",color="#E8900C")
ax[0].set_xticks(x); ax[0].set_xticklabels(arms,fontsize=8.5,rotation=12)
ax[0].set_ylabel("forecast accuracy"); ax[0].legend(fontsize=9); ax[0].grid(axis="y",alpha=.25)
ax[0].set_title("Regime ordering under both readouts",fontsize=11.5)
ax[1].scatter(tab["eff_rank"],tab["acc_GD"],s=70,color="#4C72B0",label="GD")
ax[1].scatter(tab["eff_rank"],tab["acc_ridge"],s=70,color="#E8900C",marker="s",label="ridge")
for a in arms: ax[1].annotate(a,(tab.loc[a,"eff_rank"],tab.loc[a,"acc_GD"]),fontsize=7.5,xytext=(4,3),textcoords="offset points")
ax[1].set_xlabel("effective rank (participation ratio)"); ax[1].set_ylabel("forecast accuracy")
ax[1].legend(fontsize=9); ax[1].grid(alpha=.25); ax[1].set_title("Effective rank vs accuracy",fontsize=11.5)
fig.tight_layout(); savefig(fig,"phase19_ridge_vs_gd.png")

saved /content/drive/My Drive/ChaoSSM/results/phase19_ridge_readout/phase19_ridge_vs_gd.png


## Rescue if Drive was not mounted

In [ ]:
import shutil
if not RESULTS.startswith("/content/drive"):
    z=shutil.make_archive("/content/phase19_results","zip",RESULTS); print("archive:",z)
    try:
        from google.colab import files; files.download(z)
    except Exception as e: print("download",z,"manually:",e)
else: print("on Drive:",RESULTS)

on Drive: /content/drive/My Drive/ChaoSSM/results/phase19_ridge_readout
